# Imports

In [2]:
import numpy as np
import matplotlib as plt
import pandas as pd
import json

In [3]:
import symspellpy
import pkg_resources
import re
from symspellpy.symspellpy import SymSpell, Verbosity
import pkg_resources
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
import string
from sklearn.feature_extraction.text import TfidfVectorizer

/var/folders/zv/4f9cw9vs6tjbvz5bh2k07w_40000gn/T/ipykernel_30176/3243796275.py:2: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


In [4]:
import numpy as np 
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity

from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import mean_squared_error


from surprise import SVD, Dataset, Reader, accuracy
from surprise.model_selection import cross_validate, train_test_split
from surprise import accuracy
from sklearn.metrics import roc_auc_score, confusion_matrix, precision_score, recall_score, ndcg_score, average_precision_score, ConfusionMatrixDisplay
import pandas as pd
import numpy as np

from surprise.model_selection import KFold

from collections import defaultdict

# Load Data

In [5]:
ratings_df = pd.read_csv('/Users/gracefujinaga/MSDS_490/Data/amazon_reviews/ratings_Musical_Instruments.csv')

In [6]:
file_path = '/Users/gracefujinaga/MSDS_490/Data/amazon_reviews/reviews_Musical_Instruments_5.json'

# read each line
reviews = []
with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        reviews.append(json.loads(line))

In [7]:
reviews_df =pd.DataFrame(reviews)

# Cleaning, Normalization, EDA

## Reviews DF

In [8]:
reviews_df.columns

Index(['reviewerID', 'asin', 'reviewerName', 'helpful', 'reviewText',
       'overall', 'summary', 'unixReviewTime', 'reviewTime'],
      dtype='object')

In [10]:
# clean the review text so that tfidf can be run
# normalize ratings across the user
sym_spell = SymSpell(max_dictionary_edit_distance=2, prefix_length=7)

dictionary_path = pkg_resources.resource_filename("symspellpy", "frequency_dictionary_en_82_765.txt")
sym_spell.load_dictionary(dictionary_path, term_index=0, count_index=1)

def clean_review(comment): 

    suggestions = sym_spell.lookup_compound(comment, max_edit_distance=2)

    for suggestion in suggestions:
        tokens = suggestion.term

    #split document into individual words
    tokens=tokens.split()
    re_punc = re.compile('[%s]' % re.escape(string.punctuation))

    # remove punctuation from each word
    tokens = [re_punc.sub('', w) for w in tokens]

    # # remove remaining tokens that are not alphabetic
    tokens = [word for word in tokens if word.isalpha()]

    # # filter out short tokens
    tokens = [word for word in tokens if len(word) > 4]

    #lowercase all words
    tokens = [word.lower() for word in tokens]

    # filter out stop words
    stop_words = set(stopwords.words('english'))
    tokens = [w for w in tokens if not w in stop_words]    

    # word stemming    
    ps=PorterStemmer()
    tokens=[ps.stem(word) for word in tokens]

    #print(tokens)
    return tokens


In [11]:
reviews_df['cleaned_review'] = reviews_df['reviewText'].apply(clean_review)


In [12]:
reviews_df['cleaned_review_string'] = reviews_df['cleaned_review'].apply(lambda x: " ".join(x))

In [136]:
reviews_df['user_mean'] = reviews_df.groupby('reviewerID')['overall'].transform('mean')
reviews_df['rating_centered'] = reviews_df['overall'] - reviews_df['user_mean']

## Ratings 

In [15]:
ratings_df.head()

,userId,asin,rating,timestamp
0,A1YS9MDZP93857,0006428320,3.0,1394496000
1,A3TS466QBAWB9D,0014072149,5.0,1370476800
2,A3BUDYITWUSIS7,0041291905,5.0,1381708800
3,A19K10Z0D2NTZK,0041913574,5.0,1285200000
4,A14X336IB4JD89,0201891859,1.0,1350432000


In [16]:
# Normalize ratings
ratings_df['user_mean'] = ratings_df.groupby('userId')['rating'].transform('mean')
ratings_df['rating_centered'] = ratings_df['rating'] - ratings_df['user_mean']

# Model

In [42]:
# SVD
reader = Reader(rating_scale=(1, 5))

# svd automatically normalizes the data
data = Dataset.load_from_df(ratings_df[['userId', 'asin', 'rating']], reader)
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

svd = SVD()
svd.fit(trainset)
predictions_svd = svd.test(testset)
pred_df = pd.DataFrame(predictions_svd, columns=['userId', 'asin', 'true_rating', 'pred_rating', 'details'])

In [22]:
len(pred_df)

100036

In [23]:
# # TFIDF embedding
Tfidf=TfidfVectorizer(ngram_range=(1,1))
TFIDF_embeddings=Tfidf.fit_transform(reviews_df['cleaned_review_string'].tolist())    
cosine_sim = cosine_similarity(TFIDF_embeddings, TFIDF_embeddings) 
asin_to_index = pd.Series(reviews_df.index, index=reviews_df['asin']).to_dict()

In [137]:
def get_content_score(user_id, asin):
    # find items user liked
    user_items = reviews_df[(reviews_df['reviewerID'] == user_id) & (reviews_df['rating_centered'] >= 0)]

    # if the item is not in the index skip
    if user_items.empty or asin not in asin_to_index:
        return None

    target_idx = asin_to_index[asin]
    similarities = []

    # loop through asins
    for liked_asin in user_items['asin']:
        if liked_asin not in asin_to_index or liked_asin == asin:
            continue
        liked_idx = asin_to_index[liked_asin]
        similarity = cosine_sim[target_idx][liked_idx]
        similarities.append(similarity)

    if similarities:
        return np.mean(similarities)  # Average similarity with liked items
    return None


# Exploring Reviews_df and ratings_df

In [27]:
len(reviews_df)

10261

In [28]:
df1 = reviews_df.rename(columns={'reviewerID': 'user_id'})
df2 = ratings_df.rename(columns={'userId': 'user_id'})

# Convert both to sets of tuples
pairs1 = set(zip(df1['user_id'], df1['asin']))
pairs2 = set(zip(df2['user_id'], df2['asin']))

# Find intersection
overlap = pairs1.intersection(pairs2)

# get the intersection of asin
set1 = set(zip(df1['asin']))
set2 = set(zip(df2['asin']))

# Find intersection
overlap = set1.intersection(set2)

# Count overlaps
print(f"Number of overlapping asin: {len(overlap)}")

# get the intersection of asin
set1 = set(zip(df1['user_id']))
set2 = set(zip(df2['user_id']))

# Find intersection
overlap = set1.intersection(set2)

# Count overlaps
print(f"Number of overlapping user ids: {len(overlap)}")

Number of overlapping asin: 900
Number of overlapping user ids: 1429


In [29]:
len(pairs2)

# 900 asin
# 1429 user ids

500176

# Combine predictions

In [187]:
def combine_predictions(user_id, asin, svd_predictions, alpha):
    # SVD prat
    try:
        svd_pred = svd_predictions.loc[(svd_predictions['userId'] == user_id) & (svd_predictions['asin'] == asin)]
        svd_pred_value = svd_pred['pred_rating'].values[0]
    except:
        #print(f'svd pred none for asin:{asin} and user:{user_id}')
        return None
 
    # content based part
    try: 
        content_pred_value = get_content_score(user_id, asin)  # maps 0 to 1, and 1 to 5
        #print(content_pred_value)
    except:
        #print(f'tfidf none for asin:{asin} and user:{user_id}')
        return None
    

    # combine
    if svd_pred_value is None or content_pred_value is None:
        return None
    else:
        # print(f"svd pred value: {svd_pred_value}")
        # print(f"content pred value: {content_pred_value}")
        return alpha * svd_pred_value + (1 - alpha) * content_pred_value


In [ ]:
## TRIED TO DO ALL PAIRS ultimately ended up with the same predictions that is the intersection of 
# the predictions of both the SVD model and the TF-IDF model


# users = pred_df['userId'].unique()
# items = pred_df['asin'].unique()

# # Create a DataFrame for the cross product
# user_item_pairs = pd.MultiIndex.from_product([users, items], names=['user_id', 'asin']).to_frame(index=False)

# print(user_item_pairs.head())  # Preview first few rows
# print(f"Total possible (user_id, asin) pairs: {len(user_item_pairs)}")

In [139]:
user_item_pairs = reviews_df[['reviewerID', 'asin']]

In [188]:
def predict_user_item_pairs(alpha):
    # predict for user item pairs
    predictions = []
    for _, row in user_item_pairs.iterrows():
        user_id = row['reviewerID']
        asin = row['asin']
        
        # get the combined prediction
        combined_pred = combine_predictions(user_id, asin, pred_df, alpha)
        
        # append to the df
        if combined_pred is not None:
            predictions.append({
                'userId': user_id,
                'asin': asin,
                'predicted_rating': combined_pred
            })

    # create df
    return pd.DataFrame(predictions) 

#### alpha = 0.5

In [189]:
predictions_df_1 = predict_user_item_pairs(alpha=0.5)

#### alpha = 0.7

In [190]:
predictions_df_2 = predict_user_item_pairs(alpha=0.7)

In [191]:
predictions_df_1['actual_rating'] = reviews_df['overall']
predictions_df_2['actual_rating'] = reviews_df['overall']

# Metrics and Output

In [192]:
def precision_at_k_df(df, k=10, threshold=2.5):
    user_est_true = defaultdict(list)

    # Collect predictions per user
    for _, row in df.iterrows():
        uid = row['userId']
        pred = row['predicted_rating']
        actual = row['actual_rating']
        user_est_true[uid].append((pred, actual))

    precisions = dict()
    for uid, user_ratings in user_est_true.items():
        # Sort ratings by predicted score
        user_ratings.sort(key=lambda x: x[0], reverse=True)

        # Top-K items
        top_k = user_ratings[:k]

        # Count how many of top-k are relevant
        n_rec_k = sum(est >= threshold for est, _ in top_k)
        n_rel_and_rec_k = sum((est >= threshold and true_r >= threshold) for est, true_r in top_k)

        precisions[uid] = n_rel_and_rec_k / n_rec_k if n_rec_k != 0 else 0

    return precisions


def recall_at_k_df(df, k=10, threshold=2.5):
    user_est_true = defaultdict(list)

    # Collect predictions per user
    for _, row in df.iterrows():
        uid = row['userId']
        pred = row['predicted_rating']
        actual = row['actual_rating']
        user_est_true[uid].append((pred, actual))

    recalls = dict()
    for uid, user_ratings in user_est_true.items():
        user_ratings.sort(key=lambda x: x[0], reverse=True)
        top_k = user_ratings[:k]

        n_rel = sum(true_r >= threshold for _, true_r in user_ratings)
        n_rel_and_rec_k = sum((est >= threshold and true_r >= threshold) for est, true_r in top_k)

        recalls[uid] = n_rel_and_rec_k / n_rel if n_rel != 0 else 0

    return recalls


In [193]:
def prediction_metrics(predictions_df):
    # precision @k 
    precisions = precision_at_k_df(predictions_df, k=10, threshold=2.5)
    prec_at_k = sum(prec for prec in precisions.values()) / len(precisions)
    print(f"precision at k: {prec_at_k}")

    # recall
    recalls = recall_at_k_df(predictions_df, k=10, threshold=2.5)
    rec_at_k = sum(rec for rec in recalls.values()) / len(recalls)
    print(f"recall at k: {rec_at_k}")

    # ncdg
    ncdg = ndcg_score([predictions_df['actual_rating']], [predictions_df['predicted_rating']])  
    print(f"ncdg: {ncdg}")

    # # MAP
    predictions_df['predicted_binary'] = (predictions_df['predicted_rating'] > 2.5).astype(int)
    predictions_df['actual_binary'] = (predictions_df['actual_rating'] > 2.5).astype(int)

    map = average_precision_score(predictions_df['predicted_binary'], predictions_df['actual_binary'])
    print(f"MAP: {map}")

    # auc
    auc = roc_auc_score(predictions_df['predicted_binary'], predictions_df['actual_binary'])
    print(f"AUC: {auc}")

    # RMSE
    mse = mean_squared_error(predictions_df['actual_rating'], predictions_df['predicted_rating'])
    rmse = np.sqrt(mse)
    print(f"rmse: {rmse}")


In [194]:
print("alpha = 0.5: ")
prediction_metrics(predictions_df_1)

print("\n-----------------------------\n")

print("alpha = 0.7: ")
prediction_metrics(predictions_df_2)

alpha = 0.5: 
precision at k: 0.12362637362637363
recall at k: 0.07584092494806781
ncdg: 0.9874290311032962
MAP: 0.08784762766184027
AUC: 0.5075183629242124
rmse: 2.438450026081225

-----------------------------

alpha = 0.7: 
precision at k: 0.952841008198151
recall at k: 0.9577885738600024
ncdg: 0.9873106194269003
MAP: 0.975544750952368
AUC: 0.4946230158730159
rmse: 1.6446190746054958


### Model Separated Metrics

In [195]:
def separated_predictions(user_id, asin, svd_predictions):
    # SVD prat
    try:
        svd_pred = svd_predictions.loc[(svd_predictions['userId'] == user_id) & (svd_predictions['asin'] == asin)]
        svd_pred_value = svd_pred['pred_rating'].values[0]
    except:
        #print(f'svd pred none for asin:{asin} and user:{user_id}')
        return None
 
    # content based part
    try: 
        content_pred_value = get_content_score(user_id, asin)
        content_pred_value = 1 + content_pred_value * (5 - 1)  # maps 0 to 1, and 1 to 5
        #print(content_pred_value)
    except:
        #print(f'tfidf none for asin:{asin} and user:{user_id}')
        return None
    

    # combine
    if svd_pred_value is None or content_pred_value is None:
        return None
    else:
        # print(f"svd pred value: {svd_pred_value}")
        # print(f"content pred value: {content_pred_value}")
        return (svd_pred_value, content_pred_value)


In [196]:
def predict_user_item_pairs_separate(user_item_pairs):
    # predict for user item pairs
    predictions = []
    for _, row in user_item_pairs.iterrows():
        user_id = row['reviewerID']
        asin = row['asin']
        
        # get the combined prediction
        res = separated_predictions(user_id, asin, pred_df)
        
        # append to the df
        if res is not None:
            predictions.append({
                'userId': user_id,
                'asin': asin,
                'collab_prediction': res[0],
                'content_pred': res[1]
            })

    # create df
    return pd.DataFrame(predictions) 

In [197]:
separated_pred_df = predict_user_item_pairs_separate(user_item_pairs)

In [198]:
separated_pred_df['actual_rating'] = reviews_df['overall']

collab_copy_df = separated_pred_df.copy()
collab_copy_df['predicted_rating'] = collab_copy_df['collab_prediction']

content_copy_df = separated_pred_df.copy()
content_copy_df['predicted_rating'] = content_copy_df['content_pred']

In [199]:
prediction_metrics(collab_copy_df)

precision at k: 0.96799777603349
recall at k: 0.9816526066526066
ncdg: 0.9861880384873503
MAP: 0.9995012133382449
AUC: 0.4847457627118644
rmse: 0.8977570571183688


In [207]:
def prediction_metrics_cosine(predictions_df):
    # precision @k 
    precisions = precision_at_k_df(predictions_df, k=10, threshold=1)
    prec_at_k = sum(prec for prec in precisions.values()) / len(precisions)
    print(f"precision at k: {prec_at_k}")

    # recall
    recalls = recall_at_k_df(predictions_df, k=10, threshold=1)
    rec_at_k = sum(rec for rec in recalls.values()) / len(recalls)
    print(f"recall at k: {rec_at_k}")

    # ncdg
    ncdg = ndcg_score([predictions_df['actual_rating']], [predictions_df['predicted_rating']])  
    print(f"ncdg: {ncdg}")

    # # MAP
    predictions_df['predicted_binary'] = (predictions_df['predicted_rating'] > 1).astype(int)
    predictions_df['actual_binary'] = (predictions_df['actual_rating'] > 1).astype(int)

    map = average_precision_score(predictions_df['predicted_binary'], predictions_df['actual_binary'])
    print(f"MAP: {map}")

    # auc
    auc = roc_auc_score(predictions_df['predicted_binary'], predictions_df['actual_binary'])
    print(f"AUC: {auc}")

    # RMSE
    mse = mean_squared_error(predictions_df['actual_rating'], predictions_df['predicted_rating'])
    rmse = np.sqrt(mse)
    print(f"rmse: {rmse}")


In [208]:
prediction_metrics_cosine(content_copy_df)

precision at k: 1.0
recall at k: 0.9995282495282496
ncdg: 0.9845162941380667
MAP: 0.911276402909217
AUC: 0.4990916715275299
rmse: 3.499082509766968
